<a href="https://colab.research.google.com/github/catseyehuang/Taipei-YouBike-Dashboard/blob/main/YouBike_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 「YouBike 智慧調度預測模型與預警系統」
待解決問題、需求或痛點
1. 通勤早晚尖峰時段，捷運站、校園及住宅區周邊常出現「無車可借」或「無位可還」情形。
2. YouBike 官方網站地圖及機關調度監理平台僅能顯示即時資料，未能歷史數據進行預測。
3. 租賃站無車可借或無位可還情形已持續一段時間，現行系統未具警示通知機關功能。


---



## 資料預處理與清理 (Preprocessing & Cleaning)




## 資料預處理與清理 (Data Preprocessing & Cleaning)


## 特徵工程 (Feature Engineering)




---



---



## 1. 擷取 (Extract):


### JSON 檔案讀取

In [14]:
import os
import json
import pandas as pd
from google.colab import drive
import re
import requests

In [24]:
# ==========================================
# 🚀 Data Lake JSON 批次讀取
# ==========================================

# 1. 掛載您的 Google Drive (執行時會跳出視窗要求授權)
print("🔗 正在要求授權掛載 Google Drive...")
drive.mount('/content/drive')

# 2. 指定 Data Lake 資料夾路徑
folder_path = '/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON'

all_records = []
print(f"📂 正在掃描資料夾：{folder_path}")

# 3. 掃描並解析資料夾內的所有 JSON 檔案
if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    print(f"🔍 共找到 {len(file_list)} 個 JSON 快照檔案，執行下一步開始進行清洗與對齊...")

🔗 正在要求授權掛載 Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 正在掃描資料夾：/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON
🔍 共找到 916 個 JSON 快照檔案，執行下一步開始進行清洗與對齊...


###  JSON 檔案數量統計 (7x24 網格)

In [25]:
# ==========================================
# 🚀 建立獨立的 df_file_records 支援 7x24 網格
# ==========================================

if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    # 1. 掃描資料夾內所有的 JSON 快照檔案
    json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

    file_records = []
    for filename in json_files:
        match = re.match(r'^(TPE|NTPC)_(\d{4})(\d{2})(\d{2})_(\d{2})\d{4}\.json$', filename)
        if match:
            city_code, year, month, day, hour = match.groups()

            city = 'Taipei' if city_code == 'TPE' else 'New Taipei'
            scrape_date = f"{year}-{month}-{day}"
            scrape_hour = int(hour)

            file_records.append({
                'source_file': filename,
                'city': city,
                'scrape_date': scrape_date,
                'scrape_hour': scrape_hour
            })

    df_file_records = pd.DataFrame(file_records)
    print(f"✅ 成功建立獨立的 df_file_records！共偵測到 {len(df_file_records)} 個檔案。")

    if len(df_file_records) > 0:
        # 2. 將日期轉為 datetime 格式，以萃取出「星期」
        df_file_records['scrape_date'] = pd.to_datetime(df_file_records['scrape_date'])

        # 建立星期對應表 (0=週一, 6=週日)
        weekday_map = {0: '週一', 1: '週二', 2: '週三', 3: '週四', 4: '週五', 5: '週六', 6: '週日'}
        df_file_records['weekday'] = df_file_records['scrape_date'].dt.dayofweek.map(weekday_map)

        # 3. 🎯 核心邏輯：計算每個 (城市, 星期, 小時) 組合下，有幾個「不重複的日期 (scrape_date)」
        # 這樣同一天同小時有多個檔案只算 1，但跨週的不同日期就會累加！
        weekly_hourly_counts = df_file_records.groupby(['city', 'weekday', 'scrape_hour'])['scrape_date'].nunique()

        # 將城市 (city) 展開
        df_unstacked = weekly_hourly_counts.unstack(level='city', fill_value=0)

        # 4. 建立完整的 7x24 網格索引，補齊沒有資料的小時與星期
        weekday_order = ['週一', '週二', '週三', '週四', '週五', '週六', '週日']
        all_hours = list(range(24))
        full_idx = pd.MultiIndex.from_product([weekday_order, all_hours], names=['weekday', 'scrape_hour'])

        # 重新索引以填補 0
        df_full_grid = df_unstacked.reindex(full_idx, fill_value=0)

        # ==========================================
        # 📊 呈現雙北分開的 7x24 檔案收集天數統計表 (轉置：星期為列，小時為欄)
        # ==========================================
        print("\n" + "="*45)
        print("📈 新北市 JSON 檔案收集天數統計 (7x24 星期網格)")
        print("="*45)
        # 將 scrape_hour 展開成上方欄位，並使用 reindex 強制左側列照星期一到日排序
        ntpc_pivot = df_full_grid['New Taipei'].unstack(level='scrape_hour', fill_value=0)
        ntpc_pivot = ntpc_pivot.reindex(weekday_order)
        display(ntpc_pivot)

        print("\n" + "="*45)
        print("📈 台北市 JSON 檔案收集天數統計 (7x24 星期網格)")
        print("="*45)
        tpe_pivot = df_full_grid['Taipei'].unstack(level='scrape_hour', fill_value=0)
        tpe_pivot = tpe_pivot.reindex(weekday_order)
        display(tpe_pivot)

    else:
        print("⚠️ 資料夾內無符合命名規則的 JSON 檔案。")


✅ 成功建立獨立的 df_file_records！共偵測到 916 個檔案。

📈 新北市 JSON 檔案收集天數統計 (7x24 星期網格)


scrape_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,
週一,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週二,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
週三,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
週四,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,0
週五,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週六,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週日,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



📈 台北市 JSON 檔案收集天數統計 (7x24 星期網格)


scrape_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,
週一,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週二,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
週三,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
週四,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,0
週五,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週六,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週日,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. 轉換 (Transform)

### 建立初始 DataFrame (去重複)

- 將新北市與台北市 雙北資料標準化後，合併進入同一個 DataFrame
- 根據 sno(站點編號) + update_time(站點資料更新時間)去除重複資料
- 建立站點清單：紀錄站點固定不變之資訊(城市, 行政區, 編號,名稱,總數,經緯度, 是否營業,站點資訊最後更新時間)

In [29]:
# ==========================================
# 🚀 雙北 YouBike ETL：同時建立 歷史軌跡表(df_history) 與 基礎站點清單(df_stations)
# ==========================================

# 確保欄位初始容器
all_records = []

# 1. 掃描並解析資料夾內的所有 JSON 檔案
if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    print(f"🔍 共找到 {len(file_list)} 個 JSON 快照檔案，開始進行清洗與對齊...")

    # 1. 讀取與標準化雙北欄位
    for filename in file_list:
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)

                # 🛡️ 1. 處理新北市格式 (NTPC)
                if filename.startswith("NTPC_"):
                    for item in data:
                        all_records.append({
                            'city': 'New Taipei',
                            'sno': str(item.get('sno', '')),                     # 轉字串確保對齊
                            'sna': item.get('sna', ''),
                            'sarea': item.get('sarea', ''),
                            'tot_quantity': int(item.get('tot_quantity', 0)),
                            'sbi_quantity': int(item.get('sbi_quantity', 0)),
                            'bemp': int(item.get('bemp', 0)),
                            'lat': float(item.get('lat', 0.0)),
                            'lng': float(item.get('lng', 0.0)),
                            'update_time': item.get('mday', ''),                 # 新北時間欄位為 mday
                            'is_active': int(item.get('act', 0)),
                            'source_file': filename                              # 追蹤來源檔案以供統計網格使用
                        })

                # 🛡️ 2. 處理台北市格式 (TPE)
                elif filename.startswith("TPE_"):
                    for item in data:
                        all_records.append({
                            'city': 'Taipei',
                            'sno': str(item.get('sno', '')),                     # 轉字串確保對齊
                            'sna': item.get('sna', ''),
                            'sarea': item.get('sarea', ''),
                            'tot_quantity': int(item.get('Quantity', 0)),         # 欄位對齊：Quantity -> tot_quantity
                            'sbi_quantity': int(item.get('available_rent_bikes', 0)), # 欄位對齊
                            'bemp': int(item.get('available_return_bikes', 0)),   # 欄位對齊
                            'lat': float(item.get('latitude', 0.0)),             # 欄位對齊
                            'lng': float(item.get('longitude', 0.0)),            # 欄位對齊
                            'update_time': item.get('infoTime', ''),             # 台北時間欄位為 infoTime
                            'is_active': int(item.get('act', 0)),
                            'source_file': filename                              # 追蹤來源檔案以供統計網格使用
                        })

            except Exception as e:
                print(f"⚠️ 解析檔案 {filename} 發生錯誤: {e}")

    # ==========================================
    # 🎯 階段二：Pandas 高速清洗與去重 (產出兩張主表)
    # ==========================================
    if len(all_records) > 0:
        # 建立歷史底表並轉換時間
        df_history = pd.DataFrame(all_records)

        # 1. 時間格式標準化 (強制轉為 datetime 物件，方便後續分析與排序)
        df_history['update_time'] = pd.to_datetime(df_history['update_time'], format='mixed', errors='coerce')

        # 2. 剔除無效的時間資料 (若有破壞性格式導致轉出 NaT 的話)
        df_history = df_history.dropna(subset=['update_time'])

        # 3. 🎯 核心需求：根據 sno + update_time 去除重複資料
        # 保留最後一次出現的資料 (或第一次，此處用 keep='first')
        initial_count = len(df_history)
        # 在去重前，將重複快照記錄到 df_history_duplicates 並印出
        df_history_duplicates = df_history[df_history.duplicated(subset=['sno', 'update_time'], keep=False)].sort_values(by=['sno', 'update_time'])
        if not df_history_duplicates.empty:
            print(f"\n⚠️ 發現重複快照！已記錄至 `df_history_duplicates`。以下為前 10 筆預覽：")
            display(df_history_duplicates[['city', 'sno', 'sna', 'update_time', 'tot_quantity', 'source_file']].head())
        df_history = df_history.drop_duplicates(subset=['sno', 'update_time'], keep='first').reset_index(drop=True)
        final_count = len(df_history)

        print(f"\n✅ 成功將 雙北 JSON 轉換並標準化！")
        print(f"🔄 去重複前：{initial_count} 筆 -> 依 [sno + update_time] 去重後：{final_count} 筆 (共過濾掉 {initial_count - final_count} 筆重複站點快照)。")

        # 4.  建立衍生時間欄位 (支援後續 7x24 網格與分析)
        # 提取 scrape_timestamp (來自檔名)
        temp_scrape_dt_str = df_history['source_file'].str.extract(r'_(\d{8}_\d{6})\.json', expand=False)
        #    將提取的字串轉換為 datetime 物件
        df_history['scrape_timestamp'] = pd.to_datetime(temp_scrape_dt_str, format='%Y%m%d_%H%M%S', errors='coerce')

        # scrape_date (date object: 'YYYY-MM-DD')
        # 從 scrape_timestamp 提取日期部分
        df_history['scrape_date'] = df_history['scrape_timestamp'].dt.date

        # date (date object: 'YYYY-MM-DD', 來自 update_time)
        # 從 update_time 欄位中提取日期部分
        df_history['date'] = df_history['update_time'].dt.date

        # hour (int: '0-23', 來自 update_time)
        # 從 update_time 欄位中提取小時部分
        df_history['hour'] = df_history['update_time'].dt.hour

        # weekday (int: '0-6', 來自 update_time, 0=Monday)
        # 從 update_time 欄位中提取星期幾部分
        df_history['weekday'] = df_history['update_time'].dt.dayofweek # 0=週一, 6=週日

        print(f"✅ 成功建立 歷史軌跡主表 `df_history` (共 {len(df_history)} 筆資料)。")

       # ------------------------------------------
        # 🌟 同場加映：即刻精煉出 站點維度清單 `df_stations`
        # ------------------------------------------
        # 先確保歷史資料時序正確，再取每個站點的最後一筆(最新狀態)
        df_sorted = df_history.sort_values(by=['sno', 'update_time'])
        station_columns = ['city', 'sarea', 'sno', 'sna', 'tot_quantity', 'lat', 'lng', 'is_active', 'update_time']

        df_stations = df_sorted[station_columns].drop_duplicates(subset=['sno'], keep='last').reset_index(drop=True)
        df_stations = df_stations.rename(columns={
            'tot_quantity': 'total_capacity',
            'update_time': 'last_reported_time'
        })
        print(f"✅ 成功建立站點清單,共 {len(df_stations)} 個雙北站點")
        print("="*45)
        #display(df_stations.info())
        display(df_stations.head())
        print("="*45)
        print(f"✅ 成功將建立歷史軌跡清單，共 {len(df_history)} 筆資料。")

        print("="*45)
        #display(df_history.info())
        display(df_history.head())
        print("="*45)
    else:
        print("\n⚠️ 目錄中沒有符合格式的資料，請確認雲端硬碟內是否有 JSON 檔案。")

🔍 共找到 918 個 JSON 快照檔案，開始進行清洗與對齊...

⚠️ 發現重複快照！已記錄至 `df_history_duplicates`。以下為前 10 筆預覽：


,city,sno,sna,update_time,tot_quantity,source_file
124357,Taipei,500101001,YouBike2.0_捷運科技大樓站,2026-06-16 16:00:03,28,TPE_20260616_160105.json
127676,Taipei,500101001,YouBike2.0_捷運科技大樓站,2026-06-16 16:00:03,28,TPE_20260616_160605.json
134314,Taipei,500101001,YouBike2.0_捷運科技大樓站,2026-06-16 16:14:02,28,TPE_20260616_161605.json
137633,Taipei,500101001,YouBike2.0_捷運科技大樓站,2026-06-16 16:14:02,28,TPE_20260616_162105.json
406472,Taipei,500101001,YouBike2.0_捷運科技大樓站,2026-06-16 23:25:03,28,TPE_20260616_232605.json



✅ 成功將 雙北 JSON 轉換並標準化！
🔄 去重複前：1523421 筆 -> 依 [sno + update_time] 去重後：682271 筆 (共過濾掉 841150 筆重複站點快照)。
✅ 成功建立 歷史軌跡主表 `df_history` (共 682271 筆資料)。
✅ 成功建立站點清單,共 3319 個雙北站點


,city,sarea,sno,sna,total_capacity,lat,lng,is_active,last_reported_time
0,Taipei,大安區,500101001,YouBike2.0_捷運科技大樓站,28,25.02605,121.54360,1,2026-06-18 22:30:04
1,Taipei,大安區,500101002,YouBike2.0_復興南路二段273號前,21,25.02565,121.54357,1,2026-06-18 22:27:03
2,Taipei,大安區,500101003,YouBike2.0_國北教大實小東側門,28,25.02429,121.54124,1,2026-06-18 22:15:04
3,Taipei,大安區,500101004,YouBike2.0_和平公園東側,11,25.02351,121.54282,1,2026-06-18 22:21:04
4,Taipei,大安區,500101005,YouBike2.0_辛亥復興路口西北側,16,25.02153,121.54299,1,2026-06-18 22:27:03


✅ 成功將建立歷史軌跡清單，共 682271 筆資料。


,city,sno,sna,sarea,tot_quantity,sbi_quantity,bemp,lat,lng,update_time,is_active,source_file,scrape_timestamp,scrape_date,date,hour,weekday
0,New Taipei,500201001,YouBike2.0_下庄市場,八里區,20,0,20,25.14678,121.39990,2026-06-16 12:38:15,1,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,2026-06-16,12,1
1,New Taipei,500201002,YouBike2.0_八里行政中心,八里區,20,1,18,25.15397,121.40721,2026-06-16 12:34:02,1,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,2026-06-16,12,1
2,New Taipei,500201003,YouBike2.0_八里中庄市場綜合大樓,八里區,28,7,21,25.15993,121.41407,2026-06-16 12:25:04,1,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,2026-06-16,12,1
3,New Taipei,500201004,YouBike2.0_大崁國小,八里區,20,3,17,25.16064,121.41938,2026-06-16 12:34:02,1,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,2026-06-16,12,1
4,New Taipei,500201006,YouBike2.0_龍形停車場,八里區,40,7,33,25.13041,121.45130,2026-06-16 12:06:02,1,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,2026-06-16,12,1


In [23]:


# =========================================================================
# 🌧️ YouBike 歷史資料時間對齊 + 動態氣象特徵串接 (完整版)
# =========================================================================

try:
    # 1. 🕒 時間對齊 (Time Alignment)
    # 將不規則的 update_time 齊頭捨去到最近的「小時」 (修正警告：使用小寫 'h')
    df_history['datetime_hour'] = df_history['update_time'].dt.floor('h')
    print("✅ 成功建立時間對齊欄位 `datetime_hour`。")

    # 2. 🌤️ 動態計算時間邊界 (排除停業或壞掉站點的極端舊時間)
    # 使用實際爬蟲收集檔案的日期 (scrape_date) 來錨定範圍，這是最安全的作法
    start_date = df_history['scrape_date'].min().strftime('%Y-%m-%d')
    end_date = df_history['scrape_date'].max().strftime('%Y-%m-%d')

    print(f"📡 正在依據您的 JSON 檔案區間，向 Open-Meteo 請求 {start_date} 至 {end_date} 的歷史氣象...")

    # 3. 🌐 呼叫氣象 API (暫以雙北中心點 25.03, 121.56 為大環境代表)
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude=25.03&longitude=121.56&"
           f"start_date={start_date}&end_date={end_date}&"
           f"hourly=temperature_2m,precipitation,weathercode,windspeed_10m&"
           f"timezone=Asia%2FTaipei")

    response = requests.get(url)
    response.raise_for_status() # 若 API 噴錯會直接跳到 except
    weather_data = response.json()

    # 4. 🗃️ 整理氣象資料表
    df_weather = pd.DataFrame({
        'datetime_hour': pd.to_datetime(weather_data['hourly']['time']),
        'temperature': weather_data['hourly']['temperature_2m'],
        'precipitation': weather_data['hourly']['precipitation'], # 降雨量 (mm)
        'weather_code': weather_data['hourly']['weathercode'],    # 天氣狀態碼
        'wind_speed': weather_data['hourly']['windspeed_10m']     # 風速
    })

    # 🌟 特徵工程預熱：建立「是否下雨 (is_raining)」二元標籤 (0:沒雨, 1:有雨)
    df_weather['is_raining'] = (df_weather['precipitation'] > 0).astype(int)

    print("✅ 氣象特徵表 `df_weather` 建立完畢！")

    # 5. 🔗 透過 Left Join 將氣象合併回 YouBike 主表
    # 如果先前已經合併過，先移除舊的氣象欄位避免欄位重複 (如 temperature_x, temperature_y)
    cols_to_drop = [c for c in ['temperature', 'precipitation', 'weather_code', 'wind_speed', 'is_raining'] if c in df_history.columns]
    if cols_to_drop:
        df_history = df_history.drop(columns=cols_to_drop)

    df_history = pd.merge(df_history, df_weather, on='datetime_hour', how='left')

    print("\n🎉 雙北 YouBike 歷史資料 與 氣象特徵 完美融合！")
    print("=" * 70)
    # 抽樣檢查融合成果
    check_cols = ['city', 'sno', 'sna', 'update_time', 'datetime_hour', 'sbi_quantity', 'temperature', 'precipitation', 'is_raining']
    display(df_history[check_cols].head(10))
    print("=" * 70)

except Exception as e:
    print(f"❌ 執行預處理或氣象合併時發生錯誤：{e}")

✅ 成功建立時間對齊欄位 `datetime_hour`。
📡 正在依據您的 JSON 檔案區間，向 Open-Meteo 請求 2026-06-16 至 2026-06-18 的歷史氣象...
✅ 氣象特徵表 `df_weather` 建立完畢！

🎉 雙北 YouBike 歷史資料 與 氣象特徵 完美融合！


,city,sno,sna,update_time,datetime_hour,sbi_quantity,temperature,precipitation,is_raining
0,New Taipei,500201001,YouBike2.0_下庄市場,2026-06-16 12:38:15,2026-06-16 12:00:00,0,32.5,0.1,1.0
1,New Taipei,500201002,YouBike2.0_八里行政中心,2026-06-16 12:34:02,2026-06-16 12:00:00,1,32.5,0.1,1.0
2,New Taipei,500201003,YouBike2.0_八里中庄市場綜合大樓,2026-06-16 12:25:04,2026-06-16 12:00:00,7,32.5,0.1,1.0
3,New Taipei,500201004,YouBike2.0_大崁國小,2026-06-16 12:34:02,2026-06-16 12:00:00,3,32.5,0.1,1.0
4,New Taipei,500201006,YouBike2.0_龍形停車場,2026-06-16 12:06:02,2026-06-16 12:00:00,7,32.5,0.1,1.0
5,New Taipei,500201007,YouBike2.0_十三行博物館,2026-06-16 12:33:04,2026-06-16 12:00:00,12,32.5,0.1,1.0
6,New Taipei,500201008,YouBike2.0_八里區民活動中心,2026-06-16 12:12:04,2026-06-16 12:00:00,5,32.5,0.1,1.0
7,New Taipei,500201009,YouBike2.0_挖仔尾自然生態保留區,2026-06-16 12:38:02,2026-06-16 12:00:00,2,32.5,0.1,1.0
8,New Taipei,500201010,YouBike2.0_文昌公園,2026-06-16 12:19:03,2026-06-16 12:00:00,10,32.5,0.1,1.0
9,New Taipei,500201011,YouBike2.0_八里國小,2026-06-16 11:15:04,2026-06-16 11:00:00,4,31.5,0.0,0.0


In [31]:
# =========================================================================
# 🗺️ 步驟一：計算雙北各行政區的中心經緯度
# =========================================================================
print("🔄 正在計算各行政區中心經緯度...")
# 依據城市與行政區群組，計算經緯度的平均值 (Mean)
df_sarea_centers = df_stations.groupby(['city', 'sarea'])[['lat', 'lng']].mean().reset_index()

print(f"✅ 成功鎖定雙北共 {len(df_sarea_centers)} 個行政區的空間中心點。")
display(df_sarea_centers.head(42))

🔄 正在計算各行政區中心經緯度...
✅ 成功鎖定雙北共 42 個行政區的空間中心點。


,city,sarea,lat,lng
0,New Taipei,三峽區,24.936580,121.371913
1,New Taipei,三芝區,25.256873,121.497882
2,New Taipei,三重區,25.067676,121.487083
3,New Taipei,中和區,24.997630,121.495036
4,New Taipei,五股區,25.086131,121.446371
5,New Taipei,八里區,25.153016,121.412298
6,New Taipei,土城區,24.977076,121.446452
7,New Taipei,坪林區,24.935545,121.710155
8,New Taipei,平溪區,25.029892,121.745314
9,New Taipei,新店區,24.970256,121.529696


In [37]:
# ==========================================
# 🌤️ 手動指定單日氣象抓取 (維持 1 小時顆粒度)
# ==========================================

# 1. 設定目標日期與存檔路徑
target_date = '2026-06-19' # ✏️ 之後您可以隨時修改這裡來補抓資料
output_dir = '/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/Weather_CSV'
os.makedirs(output_dir, exist_ok=True) # 如果資料夾不存在會自動建立

file_name = f'sarea_weather_{target_date.replace("-", "")}.csv'
file_path = os.path.join(output_dir, file_name)

all_sarea_weather = []
print(f"📡 開始抓取 {target_date} 各行政區氣象 (每小時一筆)...")

# 假設 df_sarea_centers (42個行政區) 已經在記憶體中
for idx, row in df_sarea_centers.iterrows():
    city, sarea, lat, lng = row['city'], row['sarea'], row['lat'], row['lng']

    # 呼叫 API
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={lat}&longitude={lng}&"
           f"start_date={target_date}&end_date={target_date}&"
           f"hourly=temperature_2m,precipitation,weathercode,windspeed_10m&"
           f"timezone=Asia%2FTaipei")

    try:
        res = requests.get(url)
        res.raise_for_status()
        data = res.json()

        # 建立小時級別的 DataFrame
        df_temp = pd.DataFrame({
            'datetime_hour': pd.to_datetime(data['hourly']['time']),
            'city': city,
            'sarea': sarea,
            'temperature': data['hourly']['temperature_2m'],
            'precipitation': data['hourly']['precipitation'],
            'weather_code': data['hourly']['weathercode'],
            'wind_speed': data['hourly']['windspeed_10m']
        })

        all_sarea_weather.append(df_temp)

    except Exception as e:
        print(f"  ❌ 抓取 [{sarea}] 氣象失敗: {e}")

# 2. 合併並存檔
if all_sarea_weather:
    df_final = pd.concat(all_sarea_weather, ignore_index=True)
    df_final['is_raining'] = (df_final['precipitation'] > 0).astype(int)

    # 輸出 CSV
    df_final.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 成功！{target_date} 氣象已儲存至：\n📂 {file_path}")
    print(f"📊 總筆數 (42行政區 × 24小時)：{len(df_final)} 筆")
    display(df_final.head(5))

📡 開始抓取 2026-06-19 各行政區氣象 (每小時一筆)...

✅ 成功！2026-06-19 氣象已儲存至：
📂 /content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/Weather_CSV/sarea_weather_20260619.csv
📊 總筆數 (42行政區 × 24小時)：1008 筆


,datetime_hour,city,sarea,temperature,precipitation,weather_code,wind_speed,is_raining
0,2026-06-19 00:00:00,New Taipei,三峽區,25.4,0.0,2,3.1,0
1,2026-06-19 01:00:00,New Taipei,三峽區,25.1,0.0,3,2.2,0
2,2026-06-19 02:00:00,New Taipei,三峽區,24.6,0.0,2,0.7,0
3,2026-06-19 03:00:00,New Taipei,三峽區,24.2,0.0,1,1.6,0
4,2026-06-19 04:00:00,New Taipei,三峽區,23.6,0.0,1,1.5,0


In [32]:



# =========================================================================
# 📡 步驟二：動態抓取各行政區專屬的 7x24 歷史天氣資料
# =========================================================================
# 自動錨定您的 YouBike JSON 資料時間範圍
start_date = df_history['scrape_date'].min().strftime('%Y-%m-%d')
end_date = df_history['scrape_date'].max().strftime('%Y-%m-%d')

all_sarea_weather = []

print(f"\n📡 開始依據行政區坐標，分批下載 {start_date} 至 {end_date} 的精準氣象...")

for idx, row in df_sarea_centers.iterrows():
    city = row['city']
    sarea = row['sarea']
    lat = row['lat']
    lng = row['lng']

    print(f"  -> 正在抓取 [{city} {sarea}] (Lat: {lat:.4f}, Lng: {lng:.4f})...")

    # 帶入該行政區專屬的經緯度
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={lat}&longitude={lng}&"
           f"start_date={start_date}&end_date={end_date}&"
           f"hourly=temperature_2m,precipitation,weathercode,windspeed_10m&"
           f"timezone=Asia%2FTaipei")

    try:
        response = requests.get(url)
        response.raise_for_status()
        weather_data = response.json()

        # 建立該行政區的小天氣表
        df_temp_weather = pd.DataFrame({
            'datetime_hour': pd.to_datetime(weather_data['hourly']['time']),
            'city': city,
            'sarea': sarea,
            'temperature': weather_data['hourly']['temperature_2m'],
            'precipitation': weather_data['hourly']['precipitation'],
            'weather_code': weather_data['hourly']['weathercode'],
            'wind_speed': weather_data['hourly']['windspeed_10m']
        })

        all_sarea_weather.append(df_temp_weather)

    except Exception as e:
        print(f"  ❌ 抓取 [{sarea}] 氣象失敗，原因: {e}")

# 合併所有行政區的天氣資料
df_sarea_weather = pd.concat(all_sarea_weather, ignore_index=True)
df_sarea_weather['is_raining'] = (df_sarea_weather['precipitation'] > 0).astype(int)

# =========================================================================
# 💾 步驟三：將歷史天氣預存至雲端硬碟 (Data Cold Storage)
# =========================================================================
# 設定與您 YouBike_Raw_JSON 相同的父目錄
output_dir = '/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike'
weather_csv_path = os.path.join(output_dir, 'history_sarea_weather.csv')

# 匯出成 CSV
df_sarea_weather.to_csv(weather_csv_path, index=False, encoding='utf-8-sig')
print(f"\n💾 [重要資產備份] 歷史天氣資料已成功預存至：\n📂 {weather_csv_path}")
print(f"📊 天氣觀測總筆數 (行政區 × 小時)：{len(df_sarea_weather)} 筆。")

# =========================================================================
# 🔗 步驟四：根據 [行政區 sarea + 小時 datetime_hour] 重新與 YouBike 主表合併
# =========================================================================
print("\n🔗 正在根據「行政區」進行多對多精準特徵融合...")

# 如果主表先前有單一台北市中心的氣象欄位，先予以清除，避免欄位名稱衝突
cols_to_drop = [c for c in ['temperature', 'precipitation', 'weather_code', 'wind_speed', 'is_raining', 'datetime_hour'] if c in df_history.columns]
if cols_to_drop:
    df_history = df_history.drop(columns=cols_to_drop)

# 重新為 df_history 補上小時對齊欄位 (使用小寫 'h')
df_history['datetime_hour'] = df_history['update_time'].dt.floor('h')

# 🌟 精準合併：這次我們同時用 'sarea' 和 'datetime_hour' 來 Join！
df_history = pd.merge(
    df_history,
    df_sarea_weather,
    on=['sarea', 'datetime_hour'],
    how='left'
)

# 補齊因 merge 產生的可能 city_y 雜訊 (保留原本的 city)
if 'city_x' in df_history.columns:
    df_history = df_history.rename(columns={'city_x': 'city'}).drop(columns=['city_y'])

print("🎉 雙北分區氣象特徵融合完畢！每一筆 YouBike 站點紀錄都已對齊「當下、當地」的真實天氣。")
print("=" * 80)
# 抽樣印出不同行政區的資料檢查看看
display(df_history[['city', 'sarea', 'sna', 'update_time', 'temperature', 'precipitation', 'is_raining']].sample(10))


📡 開始依據行政區坐標，分批下載 2026-06-16 至 2026-06-18 的精準氣象...
  -> 正在抓取 [New Taipei 三峽區] (Lat: 24.9366, Lng: 121.3719)...
  -> 正在抓取 [New Taipei 三芝區] (Lat: 25.2569, Lng: 121.4979)...
  -> 正在抓取 [New Taipei 三重區] (Lat: 25.0677, Lng: 121.4871)...
  -> 正在抓取 [New Taipei 中和區] (Lat: 24.9976, Lng: 121.4950)...
  -> 正在抓取 [New Taipei 五股區] (Lat: 25.0861, Lng: 121.4464)...
  -> 正在抓取 [New Taipei 八里區] (Lat: 25.1530, Lng: 121.4123)...
  -> 正在抓取 [New Taipei 土城區] (Lat: 24.9771, Lng: 121.4465)...
  -> 正在抓取 [New Taipei 坪林區] (Lat: 24.9355, Lng: 121.7102)...
  -> 正在抓取 [New Taipei 平溪區] (Lat: 25.0299, Lng: 121.7453)...
  -> 正在抓取 [New Taipei 新店區] (Lat: 24.9703, Lng: 121.5297)...
  -> 正在抓取 [New Taipei 新莊區] (Lat: 25.0404, Lng: 121.4447)...
  -> 正在抓取 [New Taipei 板橋區] (Lat: 25.0151, Lng: 121.4627)...
  -> 正在抓取 [New Taipei 林口區] (Lat: 25.0758, Lng: 121.3760)...
  -> 正在抓取 [New Taipei 樹林區] (Lat: 24.9828, Lng: 121.4102)...
  -> 正在抓取 [New Taipei 永和區] (Lat: 25.0090, Lng: 121.5144)...
  -> 正在抓取 [New Taipei 汐止區] (Lat: 25.0692, Lng: 12

,city,sarea,sna,update_time,temperature,precipitation,is_raining
227680,New Taipei,板橋區,YouBike2.0_捷運板橋站(3號出口),2026-06-17 06:19:02,25.1,0.4,1.0
583980,New Taipei,板橋區,YouBike2.0_忠孝國中地下停車場,2026-06-18 13:54:03,31.5,2.9,1.0
286476,Taipei,內湖區,YouBike2.0_康寧星雲街口,2026-06-17 09:11:03,29.8,0.0,0.0
44192,New Taipei,汐止區,YouBike2.0_秀峰國小(仁愛路106巷),2026-06-16 14:51:02,29.6,0.8,1.0
225229,Taipei,中正區,YouBike2.0_臺大醫院兒童醫院,2026-06-17 06:05:04,24.5,1.0,1.0
410147,New Taipei,新莊區,YouBike2.0_西盛館,2026-06-17 19:58:03,25.9,0.0,0.0
481463,New Taipei,中和區,YouBike2.0_秀朗路三段10巷12弄,2026-06-18 06:15:21,26.2,0.0,0.0
394314,Taipei,信義區,YouBike2.0_松仁路95巷口,2026-06-17 18:49:03,26.6,0.7,1.0
659778,New Taipei,三重區,YouBike2.0_仁愛仁和街口,2026-06-18 20:45:05,26.6,0.0,0.0
59826,Taipei,大安區,YouBike2.0_大安運動中心,2026-06-16 16:00:03,26.2,2.3,1.0


In [ ]:
# =========================================================================
# 🔗 步驟四：根據 [行政區 sarea + 小時 datetime_hour] 重新與 YouBike 主表合併
# =========================================================================
print("\n🔗 正在根據「行政區」進行多對多精準特徵融合...")

# 如果主表先前有單一台北市中心的氣象欄位，先予以清除，避免欄位名稱衝突
cols_to_drop = [c for c in ['temperature', 'precipitation', 'weather_code', 'wind_speed', 'is_raining', 'datetime_hour'] if c in df_history.columns]
if cols_to_drop:
    df_history = df_history.drop(columns=cols_to_drop)

# 重新為 df_history 補上小時對齊欄位 (使用小寫 'h')
df_history['datetime_hour'] = df_history['update_time'].dt.floor('h')

# 🌟 精準合併：這次我們同時用 'sarea' 和 'datetime_hour' 來 Join！
df_history = pd.merge(
    df_history,
    df_sarea_weather,
    on=['sarea', 'datetime_hour'],
    how='left'
)

# 補齊因 merge 產生的可能 city_y 雜訊 (保留原本的 city)
if 'city_x' in df_history.columns:
    df_history = df_history.rename(columns={'city_x': 'city'}).drop(columns=['city_y'])

print("🎉 雙北分區氣象特徵融合完畢！每一筆 YouBike 站點紀錄都已對齊「當下、當地」的真實天氣。")
print("=" * 80)
# 抽樣印出不同行政區的資料檢查看看
display(df_history[['city', 'sarea', 'sna', 'update_time', 'temperature', 'precipitation', 'is_raining']].sample(10))

# 3. 全域總覽儀表板 (Macro Overview Dashboard)

### 探索性資料分析 (EDA - Exploratory Data Analysis

### 雙北 YouBike 2.0 站點容量資源分佈地圖 (Station Capacity Map)

從三個空間維度來看出故事：

- 交通樞紐的「巨型大動脈」：
那些面積大到誇張的藍色圓圈，通常會極度精準地黏在捷運轉乘站、台鐵/高鐵站、以及大學城（如台大公館生活圈）的周邊。這代表微笑單車在都市計畫中，是扮演標準的「大眾運輸最後一哩路（Last-Mile Connector）」。

- 社區內部的「微血管分佈」：
相較之下，住宅區、巷弄、小公園周邊則佈滿了非常密集、但藍圈極小的「微型站（10~15 柱）」。這能展現 YouBike 2.0 無須牽電線的靈活特性，成功把服務觸角塞進城市的死角。

- 調度與營運瓶頸預警（Bottleneck Alert）：
藍色圓圈越大，代表它在通勤尖峰時間能吐出、或吞下的車流量越驚人！ 這張圖直接點名了哪些站點是未來的「調度一線戰場」。這些大圈圈只要一壞掉或被塞滿，全網的流動性就會瞬間卡死。